# ColGraphRAG WebQA — 단계별 파이프라인 튜토리얼

이 노트북은 **질의 기반 멀티모달 GraphRAG** 파이프라인을 따라갑니다.

| 단계 | 스크립트 | 역할 |
|-------|--------|------|
| 0 | `export_webqa_slice.py` *또는* 미리 만든 슬라이스 | JSONL 코퍼스 슬라이스 생성·복사 |
| 2 | `pattern.py` | 질문별 그래프 패턴 (LLM) |
| 3 | `extraction.py` | 엔티티·관계 추출 (LLM) |
| 4 | `construct.py` | NetworkX → GraphML |
| 5 | `inference.py` | ColEmbed MaxSim 검색 + Gemma 답변 |
| 6 | `eval/evaluate_webqa_qa.py` | QA-FL / QA-Acc / QA 지표 |

**사전 준비:** 아래 **환경 설정**(venv + `pip install -r requirements.txt`)을 끝내세요. 실제 LLM·ColEmbed 실행 전에는 `util/download_models.py`로 **Hugging Face 가중치**를 받아야 합니다(아래 **Hugging Face 모델 다운로드** 참고). 실제 Gemma를 쓰려면 CUDA GPU가 필요합니다. 모델 없이 빠르게 연결만 확인하려면 파이프라인 셀에서 `--dry-run`을 사용하세요.

**작업 디렉터리:** Jupyter를 저장소 루트 `colgraphrag_webqa`에서 실행하거나, **저장소 루트 경로 잡기** 셀에서 `REPO`를 맞춥니다.

**두 가지 방식:** **한 번에 실행(End-to-end)** 으로 통째로 돌리거나, **단계별 수동(Step-by-step)** 으로 단계마다 실행합니다. 같은 세션에서 둘 다 쓰지 마세요(서로 다른 `RUN_ID`를 쓰거나 출력을 이해하는 경우만 예외).


## 가상환경과 CUDA 의존성

노트북을 열기 **전에** 로컬에서 **한 번만** 설정합니다. 명령은 저장소 루트 `colgraphrag_webqa/`(`requirements.txt`, `inference.py`, `notebook/`이 있는 폴더)에서 실행한다고 가정합니다.

### 가상환경(venv) 만들기

**Linux / macOS**

```bash
cd /path/to/colgraphrag_webqa
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip setuptools wheel
```

**Windows (PowerShell)**

```powershell
cd C:\path\to\colgraphrag_webqa
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip setuptools wheel
```

**Python 3.10 이상**을 쓰세요(3.11 권장).

### 노트북을 `.venv`에 연결하기 (VS Code / Cursor)

venv를 만든 뒤, 노트북 셀 실행에 사용할 파이썬을 고릅니다.

1. 이 `.ipynb` 파일을 엽니다.
2. 노트북 우측 상단에서 **커널 선택**(또는 명령 팔레트의 **Notebook: Select Notebook Kernel**).
3. **Python 환경**을 선택합니다(먼저 **다른 커널 선택…**이 필요할 수 있음).
4. **기존 Python 환경**에서 저장소 안 인터프리터를 고릅니다:  
   **`.venv/bin/python`** (Linux/macOS) 또는 **`.venv\Scripts\python.exe`** (Windows).  
   목록에 없으면 **인터프리터 경로 입력…**으로 직접 지정합니다.

나중에 아래 **Jupyter 커널**로 표시 이름을 등록하면 `.venv` 경로를 찾지 않고 그 커널만 선택해도 됩니다.

### venv 안에 CUDA용 PyTorch 명시 설치

venv를 **활성화**하고 `pip`를 올린 뒤, 먼저 CUDA 12.4 휠을 설치합니다(`requirements.txt`와 동일한 고정):

```bash
pip install torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124 \
  --extra-index-url https://pypi.org/simple
```

그다음 나머지 의존성(networkx, transformers 등)을 설치합니다:

```bash
pip install -r requirements.txt
```

버전이 맞으면 `pip`가 이미 설치한 CUDA PyTorch를 유지합니다. 그렇지 않으면 아래 **한 번에 설치** 방식을 권장합니다.

### `requirements.txt`로 GPU 스택 한 번에 설치

파일 상단에 `--index-url https://download.pytorch.org/whl/cu124`로 **CUDA 12.4용 PyTorch**가 고정되어 있습니다. 전체를 한 번에 설치하면 위 두 단계와 동일합니다:

```bash
pip install -r requirements.txt
```

설치 후 PyTorch가 CUDA를 보는지 확인합니다:

```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

NVIDIA 드라이버와 GPU가 있으면 빌드 이름에 `+cu124`가 보이고 `True`가 나와야 합니다.

### Jupyter 커널(선택)

이 노트북을 venv 안에서 돌리려면:

```bash
pip install ipykernel
python -m ipykernel install --user --name colgraphrag-webqa --display-name "Python (colgraphrag_webqa)"
```

VS Code / Cursor / Jupyter에서 **Python (colgraphrag-webqa)** 커널을 선택합니다.

**CPU 전용 참고:** `requirements.txt`는 CUDA 기준입니다. CPU만으로 스모크 테스트하려면 별도 고정이 필요합니다(여기서는 다루지 않음). 아래 파이프라인은 Gemma를 GPU에 올리지 않고 `--dry-run`을 지원합니다.

---


## 사용 가능한 GPU

venv를 활성화하고 PyTorch(`requirements.txt`) 설치를 마친 뒤 실행하세요. 가능하면 `nvidia-smi`로 **NVIDIA 드라이버/GPU**를 보여 주고, 이어서 `torch`로 **PyTorch CUDA** 상태를 출력합니다(torch가 아직 없으면 torch 관련 줄은 건너뜁니다).

In [1]:
import shutil
import subprocess

nv = shutil.which("nvidia-smi")
if nv:
    r = subprocess.run(
        [
            nv,
            "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if r.returncode == 0:
        print("nvidia-smi (GPU 목록):\n")
        print(r.stdout.strip() or "(비어 있음)")
    else:
        print("nvidia-smi 실패:", r.stderr)
else:
    print("PATH에 nvidia-smi 없음 — NVIDIA CLI가 없거나 GPU 머신이 아님.")

print()

try:
    import torch
except ImportError:
    print("torch: 아직 설치되지 않음 — 먼저 pip install -r requirements.txt를 완료하세요.")
else:
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("torch.cuda.device_count():", n)
        for i in range(n):
            print(f"  [{i}]", torch.cuda.get_device_name(i))
            print("      할당된 메모리 MB:", torch.cuda.memory_allocated(i) / 1e6)

nvidia-smi (GPU 목록):

0, NVIDIA RTX A6000, 570.195.03, 46068 MiB, 15656 MiB, 29841 MiB
1, NVIDIA RTX A6000, 570.195.03, 46068 MiB, 18456 MiB, 27041 MiB

torch: 2.6.0+cu124
torch.cuda.is_available(): True
torch.cuda.device_count(): 2
  [0] NVIDIA RTX A6000
      할당된 메모리 MB: 0.0
  [1] NVIDIA RTX A6000
      할당된 메모리 MB: 0.0


## 저장소 루트와 경로 맞추기

파이프라인은 `PYTHONPATH`에 저장소 루트가 들어가야 `mllm`, `util`, `prompt` 같은 import가 해석됩니다.

In [ ]:
import os
import sys
from pathlib import Path

# cwd 또는 상위 디렉터리에서 inference.py가 있는 저장소 루트를 찾음
_here = Path.cwd()
REPO = None
for _p in [_here, *list(_here.resolve().parents)]:
    if (_p / "inference.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError(
        "저장소 루트(inference.py가 있는 폴더)를 찾을 수 없습니다. "
        "Jupyter cwd를 colgraphrag 루트로 맞추거나 해당 폴더로 cd한 뒤 이 셀을 다시 실행하세요."
    )

assert (REPO / "inference.py").is_file(), f"REPO를 저장소 루트로 맞추세요; 현재 {REPO}"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("REPO =", REPO.resolve())
print("cwd =", Path.cwd())

## 환경 확인(선택)

**실제** 추론에서는 일반적인 단일 GPU 환경에서 VRAM을 줄이려고 `GEMMA4_E4B_IT_TORCH_DTYPE=bf16`을 쓰는 것이 좋습니다.

In [3]:
import torch

print("torch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("디바이스:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
CUDA 사용 가능: True
디바이스: NVIDIA RTX A6000


## 설정 파일

- `config/data.yaml` — 데이터셋 경로(`WEBQA_DATA_ROOT`, 슬라이스 위치).
- `config/model.yaml` — Gemma, ColEmbed, fluency 모델 경로 및 HF 저장소 ID.
- 필요하면 `.env.example`을 복사해 `.env`로 두고 `HF_TOKEN` 등을 넣습니다.

모델은 **먼저 저장소 안**(`models/mllm/`, `models/retriever/`)에서 찾고, 없으면 `/workspace/models/` 등으로 폴백합니다.

In [5]:
from pathlib import Path

_cfgs = [REPO / "config/data.yaml", REPO / "config/model.yaml"]
_ok = 0
for p in _cfgs:
    ex = p.is_file()
    if ex:
        _ok += 1
    print(p.name, "있음:" if ex else "없음:", p)
print("설정 파일 준비:", f"{_ok}/{len(_cfgs)}")


## Hugging Face 모델 다운로드

가중치는 git 저장소에 **포함되어 있지 않습니다**. `util/download_models.py`로 `config/model.yaml`에 적힌 체크포인트를 `models/`로 받습니다(Gemma, ColEmbed, 메트릭용 선택 fluency BART).

- 모델이 게이트되어 있으면(Gemma) 저장소 루트 **`.env`**에 **`HF_TOKEN`** 또는 **`HUGGING_FACE_HUB_TOKEN`**을 넣습니다.
- **다운로드 후 구조:** `models/mllm/gemma-4-E4B-it/`, `models/retriever/llama-nemotron-colembed-vl-3b-v2/`, `models/eval/bart-large-cnn/`.
- **CLI:** `python util/download_models.py`(전체), 또는 `--only gemma colembed`, 또는 `--dry-run`(계획 경로만 출력).

모델 이름·경로·HF 설정은 루트 **`README.md`**를 참고하세요.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Requires REPO from "Resolve repo root" above; fallback for quick runs
try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

print("=== Hugging Face 체크포인트 다운로드 (실행) ===")
print("스크립트: util/download_models.py")
print("대상: gemma, colembed (`config/model.yaml` 경로로 저장)")
print("주의: 시간이 길 수 있고, Gemma 등 게이트 모델은 `.env`의 HF_TOKEN 이 필요할 수 있습니다.")
print("아래에 huggingface_hub / 스크립트 로그가 그대로 출력됩니다.\n")

_dl_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_rc_dl = subprocess.run(
    [sys.executable, "-u", str(dl), "--only", "gemma", "colembed"],
    cwd=str(_r),
    env=_dl_env,
    check=False,
)
print("\n다운로드 프로세스 종료 코드:", _rc_dl.returncode)


## 간단한 텍스트 QA (RAG 아님)

`util/download_models.py`로 받은 **Gemma 4 E4B IT**를 올리고, **검색·GraphRAG·슬라이스 없이** 질문에 대해 답만 생성합니다. 파이프라인(패턴/추출/추론)과는 별도로, **로드 → 질문 → 답**만 확인할 때 쓰면 됩니다.

- 첫 실행은 가중치 로딩에 시간이 걸릴 수 있습니다. GPU가 있으면 셀에서 `bf16`을 기본으로 잡습니다.
- 가중치 폴더가 없으면 위 **모델 다운로드** 셀을 먼저 실행하세요.


In [ ]:
# 사전 조건: "저장소 루트와 경로 맞추기" 셀을 실행해 `REPO`가 있어야 합니다.
import logging
import os
import sys
import time
from pathlib import Path

if "REPO" not in globals():
    raise RuntimeError("먼저 REPO를 정의하는 셀을 실행하세요.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# 모델 로딩·HF 스택에서 나오는 로그를 노트북 출력으로 보이게 함
def _setup_verbose_logging() -> None:
    fmt = logging.Formatter("%(levelname)s [%(name)s] %(message)s")
    root = logging.getLogger()
    if not any(isinstance(h, logging.StreamHandler) and getattr(h, "stream", None) is sys.stdout for h in root.handlers):
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        root.addHandler(h)
    root.setLevel(logging.INFO)
    for name in ("mllm.hf_gemma_4_e4b_it", "transformers", "transformers.modeling_utils"):
        logging.getLogger(name).setLevel(logging.INFO)
    try:
        from transformers import logging as tf_logging

        tf_logging.set_verbosity_info()
    except Exception:
        pass


_setup_verbose_logging()

import torch
from util.llm_defaults import effective_gemma4_e4b_it_model_path
from mllm.hf_gemma_4_e4b_it import configured, generate_text

_gemma_dir = effective_gemma4_e4b_it_model_path()
print("Gemma 가중치 경로:", _gemma_dir)
print("torch / CUDA:", torch.__version__, "|", torch.cuda.is_available())
print("가중치 사용 가능(configured):", configured())

if not configured():
    print("가중치 디렉터리가 없습니다. 위 'Hugging Face 모델 다운로드' 셀을 실행한 뒤 다시 시도하세요.")
else:
    if torch.cuda.is_available():
        os.environ.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")
        print("dtype 힌트: GEMMA4_E4B_IT_TORCH_DTYPE =", os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE", "(default)"))

    question = "What is the capital of France? Answer in one short sentence."
    print("\n--- 질문 ---\n", question)
    print("\n--- 응답 생성 중: 첫 실행 시 아래에 모델·프로세서 로딩 로그가 찍힌 뒤 generate_text가 진행됩니다 ---\n")
    t0 = time.perf_counter()
    answer = generate_text(question, max_new_tokens=128)
    elapsed = time.perf_counter() - t0
    print("\n--- 답변 ---\n", answer)
    print(f"\n--- 끝 (wall time {elapsed:.2f}s) ---")


## 토이 데이터셋(shard 14)

`data/webqa/webqa_shard14_toy/webqa_slice/`가 이미 저장소에 있으면 **다시 만들지 않아도** 됩니다.

없으면 토이 슬라이스를 생성합니다(전체 WebQA + `scripts/build_shard14_toy_split.py` 기본값과 같은 베이스라인 파일 필요):

```bash
python scripts/build_shard14_toy_split.py
```

PNG는 `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` 등에 있거나 `webqa_images.jsonl`에 기록된 경로를 가리킵니다. 아래 **토이 이미지 미리보기**를 실행하면 노트북에서 몇 장을 볼 수 있습니다.

In [ ]:
TOY_SLICE = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

print("=== 토이 데이터셋(WebQA 슬라이스) ===")
print("데이터셋: WebQA — shard 14 **toy** 슬라이스 (미니 코퍼스, 질의·이미지·텍스트 JSONL)")
print("경로:", TOY_SLICE.resolve())
print("일반 구성: webqa_questions.jsonl(질문·정답·메타), webqa_texts.jsonl(텍스트 청크), webqa_images.jsonl(이미지 인덱스), webqa_tables.jsonl(선택)")

_names = (
    "webqa_questions.jsonl",
    "webqa_texts.jsonl",
    "webqa_images.jsonl",
    "webqa_tables.jsonl",
)
for _name in _names:
    _p = TOY_SLICE / _name
    if _p.is_file():
        _n = sum(1 for _ in _p.open(encoding="utf-8"))
        print(f"  {_name}: {_n}줄")
    else:
        print(f"  {_name}: (파일 없음)")

_qp = TOY_SLICE / "webqa_questions.jsonl"
if _qp.is_file():
    import json as _json
    from collections import Counter

    _qcates = Counter()
    _splits = Counter()
    _n_img_only = _n_txt_only = _n_both = _n_neither = 0
    _n_chars_q = 0
    _rows = 0
    for _line in _qp.open(encoding="utf-8"):
        _line = _line.strip()
        if not _line:
            continue
        _rows += 1
        _o = _json.loads(_line)
        _qcates[str(_o.get("Qcate") or "?")] += 1
        _splits[str(_o.get("split") or "?")] += 1
        _q = _o.get("Q") or ""
        _n_chars_q += len(_q)
        _hi = bool(_o.get("img_posFacts"))
        _ht = bool(_o.get("txt_posFacts"))
        if _hi and _ht:
            _n_both += 1
        elif _hi:
            _n_img_only += 1
        elif _ht:
            _n_txt_only += 1
        else:
            _n_neither += 1
    print("질문 JSONL 상세 (webqa_questions.jsonl):")
    print("  레코드 수:", _rows)
    if _rows:
        print("  질문 길이 평균(글자 수):", round(_n_chars_q / _rows, 1))
    print("  Qcate별 개수:", dict(sorted(_qcates.items())))
    print("  split별 개수:", dict(sorted(_splits.items())))
    print(
        "  긍정 증거 타입 — 이미지+텍스트:",
        _n_both,
        "| 이미지만:",
        _n_img_only,
        "| 텍스트만:",
        _n_txt_only,
        "| 둘 다 없음:",
        _n_neither,
    )
else:
    print("webqa_questions.jsonl 없음 — 위 상세 통계를 계산할 수 없습니다.")


### 토이 이미지 미리보기(선택)

`webqa_images.jsonl`을 읽어 PNG 몇 장을 표시합니다. JSONL 안 경로가 `/workspace/...`를 가리키더라도 파일이 없으면, 같은 파일명으로 저장소 안 샤드 폴더 `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/`를 시도합니다.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    _slice = TOY_SLICE
except NameError:
    try:
        _slice = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"
    except NameError:
        _here = Path.cwd()
        _root = _here if (_here / "data" / "webqa").is_dir() else _here.parent
        _slice = _root / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "inference.py").is_file() else _here.parent

imgs_jsonl = _slice / "webqa_images.jsonl"
local_shard = _repo / "data" / "webqa" / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014"

if not imgs_jsonl.is_file():
    print("없음:", imgs_jsonl)
else:
    rows = []
    with imgs_jsonl.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"슬라이스 인덱스 이미지 수: {len(rows)}")
    for obj in rows[:5]:
        url = obj.get("url") or ""
        p = Path(url)
        if not p.is_file() and p.name:
            alt = local_shard / p.name
            if alt.is_file():
                p = alt
        if not p.is_file():
            print("건너뜀(파일 없음):", url[:80], "...")
            continue
        print(obj.get("image_id"), obj.get("title", "")[:60])
        display(IPyImage(filename=str(p), width=400))

## 처음부터 끝까지 한 번에 실행(학습자 권장)

`tests/test_pipeline.py`가 올바른 환경 변수로 0→6 단계를 묶어 실행합니다. 터미널에서 돌릴 때와 같습니다.

**단계별 수동 실행**을 쓰는 경우 이 절은 **건너뛰어도** 됩니다.

- **`-n 5`** — 질문 5개(빠름).
- **`--dry-run`** — GPU LLM 없음; 자리 표시 답(배선 확인용).
- **`GEMMA4_E4B_IT_TORCH_DTYPE=bf16`** — VRAM이 부족할 때 실제 실행에 권장.

In [ ]:
import os
import subprocess
import sys

# 기본은 실추론(GPU). True면 드라이런(스모크·CPU 친화)
DRY_RUN = False  # 기본: GPU·실모델 (빠른 스모크만: True)
N_QUERIES = 2

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
if not DRY_RUN:
    env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

cmd = [
    sys.executable,
    str(REPO / "tests" / "test_pipeline.py"),
    "-n", str(N_QUERIES),
]
if DRY_RUN:
    cmd.append("--dry-run")

print("통합 파이프라인 설정  DRY_RUN=", DRY_RUN, " N_QUERIES=", N_QUERIES)
print("실행 명령:", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO), env=env)
print("종료 코드:", r.returncode)

# 통합 테스트 후 최근 result/ 폴더와 산출물 존재 여부 안내
print("\n--- 통합 실행 결과 위치(참고) ---")
_rr = REPO / "result"
if _rr.is_dir():
    _runs = sorted(_rr.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for _p in _runs[:5]:
        _pred = _p / "phase5_inference" / "predictions.json"
        _rep = _p / "phase5_inference" / "evaluation_report.json"
        _flags = []
        if _pred.is_file():
            _flags.append("predictions")
        if _rep.is_file():
            _flags.append("evaluation_report")
        _tag = ", ".join(_flags) if _flags else "(추론/평가 산출물 없음)"
        print(f"  {_p.name}  |  {_tag}")
    if _runs:
        _latest_rep = _runs[0] / "phase5_inference" / "evaluation_report.json"
        if _latest_rep.is_file():
            import json as _json

            def _sf(_x):
                try:
                    return float(_x) if _x is not None else 0.0
                except (TypeError, ValueError):
                    return 0.0

            with _latest_rep.open(encoding="utf-8") as _f:
                _repd = _json.load(_f)
            _all_lb = (_repd.get("leaderboard_summary") or {}).get("All") or {}
            if _all_lb:
                print("\n--- 가장 최근 result 폴더의 평가 점수(전체 All) ---")
                print(
                    f"  QA-FL={_sf(_all_lb.get('QA-FL')):.4f}  "
                    f"QA-Acc={_sf(_all_lb.get('QA-Acc')):.4f}  "
                    f"QA={_sf(_all_lb.get('QA')):.4f}"
                )
                print("  리포트:", _latest_rep.resolve())
else:
    print("  result/ 폴더가 없습니다. 파이프라인을 한 번 실행해 보세요.")


실행이 끝나면 결과는 **`result/<RUN_ID>/`** 아래에 생깁니다:

- `webqa_slice/` — 복사되거나 내보낸 JSONL
- `phase2_pattern_cache/`, `phase3_extraction_cache/`
- `phase4_graphs_out/*.graphml`
- `phase5_inference/predictions.json`, `evaluation_report.json`

In [ ]:
# 최근 결과 실행 목록
import datetime

result_root = REPO / "result"
if result_root.is_dir():
    runs = sorted(result_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:5]:
        ts = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        pred = (p / "phase5_inference" / "predictions.json").is_file()
        rep = (p / "phase5_inference" / "evaluation_report.json").is_file()
        print(p.name, "| 수정 시각:", ts.isoformat(sep=" ", timespec="seconds"))
        print("    predictions:", pred, "| evaluation_report:", rep)
else:
    print("아직 result/ 없음 — 먼저 파이프라인을 한 번 실행하세요.")


## GraphML 하나 살펴보기(4단계 출력)

그래프는 일반 NetworkX GraphML이며, **nodes** · **edges** 용어는 아래 출력에서 영어 그대로 둡니다. 각 node에는 `entity_name`, `type`, `description`, `source_id` 등의 속성이 붙습니다.

**이 절의 코드 셀:** `nx.read_graphml`로 첫 GraphML을 읽고, 통계를 출력한 뒤 **matplotlib**으로 spring layout **정적 미리보기**를 그립니다(노드가 너무 많으면 plot은 건너뜁니다).

**대화형 시각화:** 아래 **데모 UI(선택)** 절에서 FE를 띄우면, 브라우저에서 질문별 그래프를 **force-directed**로 탐색할 수 있습니다. 구현은 Vite 기반 FE의 `GraphViewer` 컴포넌트(`react-force-graph-2d`)가 백엔드에서 받은 nodes/edges를 그립니다.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

graphs_dir = REPO / "result"
gml_files = list(graphs_dir.glob("**/phase4_graphs_out/*.graphml"))
if not gml_files:
    print("No GraphML under result/*/phase4_graphs_out/")
else:
    path = sorted(gml_files)[0]
    G = nx.read_graphml(path)
    print("file:", path.relative_to(REPO))
    print("nodes:", G.number_of_nodes(), "  edges:", G.number_of_edges())
    try:
        _Gu = G.to_undirected() if G.is_directed() else G
        print("density (undirected):", f"{nx.density(_Gu):.6f}")
    except Exception as _e:
        print("density (undirected) skipped:", _e)
    for nid in list(G.nodes)[:3]:
        print(" sample node:", nid, dict(G.nodes[nid]))

    # quick layout (large graphs: layout can be slow)
    _G = G.to_undirected() if G.is_directed() else G
    _n = _G.number_of_nodes()
    if _n == 0:
        pass
    elif _n > 300:
        print("plot skipped: too many nodes (", _n, ") — open Gephi/GraphML tools for big graphs")
    else:
        fig, ax = plt.subplots(figsize=(7, 5), dpi=110)
        k = 0.6 / max(1, _n ** 0.5)
        pos = nx.spring_layout(_G, seed=0, k=k, iterations=50)
        nx.draw_networkx(
            _G,
            pos=pos,
            ax=ax,
            with_labels=False,
            node_size=180,
            width=0.6,
            edge_color="gray",
            alpha=0.9,
        )
        ax.set_title("GraphML preview (spring layout) — first .graphml under result/")
        ax.axis("off")
        fig.tight_layout()
        plt.show()


## 단계별 실행(수동)

다음 셀에서 `RUN_ID`, `N_QUERIES`, `DRY_RUN`을 정한 뒤 0→6 단계를 순서대로 실행합니다. `tests/test_pipeline.py`와 같은 설정(토이 데이터, 로컬 경로)입니다.

**처음부터 끝까지 한 번에** 이미 돌렸다면 이 절은 **건너뛰거나**, 여기서 **새 `RUN_ID`**를 쓰세요.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# --- 실험에서 바꿀 값 ---
RUN_ID = "notebook_tutorial_run"  # 실험마다 변경
N_QUERIES = 2
DRY_RUN = False  # 기본: GPU·Gemma 실추론 (스모크/CPU만: True)

# --- 경로(tests/test_pipeline.py와 동일 로직) ---
_LOCAL_DATA = REPO / "data" / "webqa"
if (_LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice").is_dir():
    TOY_SLICE = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice"
else:
    TOY_SLICE = Path("/workspace/data/webqa/WebQA_imgs_7z_chunks/webqa_shard14_toy/webqa_slice")

if (_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014").is_dir():
    SHARD14_IMGS = str(_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014")
else:
    SHARD14_IMGS = "/workspace/data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014"

result_dir = REPO / "result" / RUN_ID
slice_dir = result_dir / "webqa_slice"
q_file = slice_dir / "webqa_questions.jsonl"
t_file = slice_dir / "webqa_texts.jsonl"

dry_run = "1" if DRY_RUN else "0"

from util.llm_defaults import DEFAULT_GEMMA4_E4B_IT_MODEL_PATH
gemma_path = os.environ.get("GEMMA4_E4B_IT_MODEL_PATH", "").strip() or str(DEFAULT_GEMMA4_E4B_IT_MODEL_PATH)

common_env = {
    "PYTHONPATH": str(REPO),
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "WEBQA_RUN_PROFILE": "val_n100",
    "WEBQA_DATA_ROOT": str(_LOCAL_DATA) if _LOCAL_DATA.is_dir() else "/workspace/data/webqa",
    "PATTERN_MAX_SAMPLES": str(N_QUERIES),
    "WEBQA_EXPORT_MAX": str(N_QUERIES),
    "EXTRACTION_MAX_QUESTIONS": str(N_QUERIES),
    "CONSTRUCT_MAX_QUESTIONS": str(N_QUERIES),
    "PATTERN_DRY_RUN": dry_run,
    "EXTRACTION_DRY_RUN": dry_run,
    "PATTERN_CONCURRENCY": "1",
    "EXTRACTION_CONCURRENCY": "1",
    "GEMMA4_E4B_IT_MODEL_PATH": gemma_path,
    "VIDORE_TEXT_LLM_BACKEND": "hf_gemma_4_e4b_it",
    "COLEMBED_MODEL_PATH": os.environ.get(
        "COLEMBED_MODEL_PATH",
        "/workspace/models/retriever/llama-nemotron-colembed-vl-3b-v2",
    ),
}
if os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE"):
    common_env["GEMMA4_E4B_IT_TORCH_DTYPE"] = os.environ["GEMMA4_E4B_IT_TORCH_DTYPE"]
elif not DRY_RUN:
    common_env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

def run_step(desc: str, cmd: list[str], env: dict) -> int:
    merged = {**os.environ, **env}
    print(f"\n=== {desc} ===\n{' '.join(cmd)}\n")
    r = subprocess.run(cmd, cwd=str(REPO), env=merged)
    print(f"종료 코드={r.returncode}")
    return r.returncode

print("RUN_ID:", RUN_ID, "| N_QUERIES:", N_QUERIES, "| DRY_RUN:", DRY_RUN)
print("토이 슬라이스:", TOY_SLICE)

print("결과 디렉터리:", result_dir)
print("슬라이스(다음 셀에서 복사됨):", slice_dir)
print("이미지 샤드:", SHARD14_IMGS)
print("Gemma 경로:", gemma_path)
print("ColEmbed:", common_env.get("COLEMBED_MODEL_PATH"))


### 미리 만든 토이 `webqa_slice`를 `result/<RUN_ID>/`로 복사

In [ ]:
if not TOY_SLICE.is_dir():
    raise FileNotFoundError(f"토이 슬라이스 없음: {TOY_SLICE}")

result_dir.mkdir(parents=True, exist_ok=True)
if slice_dir.exists():
    shutil.rmtree(slice_dir)
shutil.copytree(TOY_SLICE, slice_dir)
print("복사 완료:", slice_dir)
_qpath = slice_dir / "webqa_questions.jsonl"
if _qpath.is_file():
    _nq = sum(1 for _ in _qpath.open(encoding="utf-8"))
    print("슬라이스 질문 줄 수(webqa_questions.jsonl):", _nq)
else:
    print("(webqa_questions.jsonl 없음)")

### 패턴 (`pattern.py`)

**여기서 말하는 패턴**은 [Query-Driven Multimodal GraphRAG (Bu et al., ACL 2025 Findings)](https://aclanthology.org/2025.findings-acl.1100/)에서 설명하는 것처럼, **질의 의미(query semantics)에 맞춰 동적으로 잡히는 지역 지식 그래프의 골격**입니다. 논문에서는 (1) 질의 의미로부터 **그래프 패턴을 도출**해 지식 추출을 안내하고, (2) 다중 경로 검색으로 핵심 지식을 찾으며, (3) 필요 시 멀티모달 정보를 보충한다고 정리합니다.

이 노트북의 2단계에서는 LLM이 질문별로 **어떤 형태의 부분 그래프를 만들지**(예: 어떤 역할의 노드·관계 타입을 우선할지)를 정하는 **패턴**을 생성합니다. 다음 3단계 추출은 그 패턴을 따라 텍스트·이미지 증거에서 엔티티와 관계를 채워 넣습니다. 요약하면 **패턴 = 질문 맞춤 그래프 설계도**, **추출 = 설계도에 맞춘 내용 채우기**에 가깝습니다.



In [ ]:
pattern_cache = result_dir / "phase2_pattern_cache"
_local_pattern = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice" / "webqa_questions.jsonl"
pattern_question_file = str(_local_pattern) if _local_pattern.is_file() else "/workspace/data/webqa/WebQA_data_first_release/WebQA_train_val.json"

pattern_env = {
    **common_env,
    "PATTERN_JSON_FILE_PATH": pattern_question_file,
    "PATTERN_CACHE_DIR": str(pattern_cache),
}
_rc_p2 = run_step("2단계 패턴", [sys.executable, "pattern.py"], pattern_env)
print("패턴 캐시 디렉터리:", pattern_cache, "| 존재:", pattern_cache.is_dir())
if _rc_p2 != 0:
    print("경고: 2단계가 비정상 종료였습니다. 종료 코드:", _rc_p2)

### 추출 (`extraction.py`)

In [ ]:
extraction_cache = result_dir / "phase3_extraction_cache"
extraction_env = {
    **common_env,
    "EXTRACTION_QUESTION_FILE": str(q_file),
    "EXTRACTION_TEXT_FILE": str(t_file),
    "EXTRACTION_PATTERN_CACHE_DIR": str(pattern_cache),
    "EXTRACTION_CACHE_DIR": str(extraction_cache),
}
_rc_p3 = run_step("3단계 추출", [sys.executable, "extraction.py"], extraction_env)
print("추출 캐시 디렉터리:", extraction_cache, "| 존재:", extraction_cache.is_dir())
if _rc_p3 != 0:
    print("경고: 3단계가 비정상 종료였습니다. 종료 코드:", _rc_p3)

### GraphML 구성 (`construct.py`)

In [ ]:
graph_dir = result_dir / "phase4_graphs_out"
construct_env = {
    **common_env,
    "CONSTRUCT_QUESTION_FILE": str(q_file),
    "CONSTRUCT_TABLE_FILE": str(slice_dir / "webqa_tables.jsonl"),
    "CONSTRUCT_IMAGE_FILE": str(slice_dir / "webqa_images.jsonl"),
    "CONSTRUCT_TEXT_FILE": str(t_file),
    "CONSTRUCT_EXTRACTION_CACHE": str(extraction_cache),
    "CONSTRUCT_OUTPUT_GRAPH_DIR": str(graph_dir),
    "CONSTRUCT_WEBQA_SLICE_DIR": str(slice_dir),
}
_rc_p4 = run_step("4단계 그래프 구성", [sys.executable, "construct.py"], construct_env)
_gml_list = list(graph_dir.glob("*.graphml")) if graph_dir.is_dir() else []
print("GraphML 출력 디렉터리:", graph_dir, "| 파일 개수:", len(_gml_list))
if _rc_p4 != 0:
    print("경고: 4단계가 비정상 종료였습니다. 종료 코드:", _rc_p4)

### 추론 (`inference.py`)

켜져 있으면 ColEmbed 검색을 쓰고, 답은 Gemma로 생성합니다(`DRY_RUN`이면 생략).

In [ ]:
phase5_dir = result_dir / "phase5_inference"
phase5_dir.mkdir(parents=True, exist_ok=True)
pred_json = phase5_dir / "predictions.json"

inference_env = {
    **common_env,
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "INFERENCE_GRAPH_DIR": str(graph_dir),
    "INFERENCE_QUESTION_FILE": str(q_file),
    "INFERENCE_OUTPUT_JSON": str(pred_json),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_DRY_RUN": dry_run,
    "INFERENCE_COLEMBED_RETRIEVAL": os.environ.get("INFERENCE_COLEMBED_RETRIEVAL", "1"),
    "INFERENCE_WEBQA_SLICE_DIR": str(slice_dir),
    "WEBQA_IMGS_DIR": SHARD14_IMGS,
}
_rc_p5 = run_step("5단계 추론", [sys.executable, "inference.py"], inference_env)

# predictions.json은 qid -> 답 문자열 맵(JSON 객체)
if pred_json.is_file():
    import json as _json

    with pred_json.open(encoding="utf-8") as _f:
        _preds = _json.load(_f)
    _keys = list(_preds.keys()) if isinstance(_preds, dict) else []
    print("\n--- 추론 결과 요약 ---")
    print("출력 파일:", pred_json)
    print("예측 qid 수:", len(_keys))
    for _qid in _keys[:5]:
        _ans = str(_preds[_qid])[:240]
        print(f"  qid={_qid}  답(앞 240자): {_ans!r}")
    if len(_keys) > 5:
        print(f"  ... 외 {len(_keys) - 5}개")
else:
    print("추론 출력 없음:", pred_json)
if _rc_p5 != 0:
    print("경고: 5단계가 비정상 종료였습니다. 종료 코드:", _rc_p5)

### QA 평가 (`eval/evaluate_webqa_qa.py`)

In [ ]:
import json
from pathlib import Path


def _print_webqa_report_summary_kr(report_path: Path) -> None:
    """evaluation_report.json에서 리더보드·원시 지표·검색 요약을 한글로 출력."""

    def _f(x) -> float:
        try:
            if x is None:
                return 0.0
            return float(x)
        except (TypeError, ValueError):
            return 0.0

    if not report_path.is_file():
        print("평가 리포트 파일 없음:", report_path)
        return
    with report_path.open(encoding="utf-8") as f:
        rep = json.load(f)
    counts = rep.get("counts") or {}
    diag = rep.get("diagnostics") or {}

    print("\n" + "=" * 62)
    print("WebQA QA 평가 점수 요약")
    print("=" * 62)
    print(
        "샘플 수 — 전체:",
        counts.get("All"),
        "| 채점된 qid:",
        counts.get("scored"),
        "| 단일모달:",
        counts.get("Unimodal"),
        "| 멀티모달:",
        counts.get("Multimodal"),
    )
    if diag.get("webqa_fluency_effective_backend"):
        print(
            "유창도 백엔드:",
            diag.get("webqa_fluency_effective_backend"),
            "| 모델:",
            diag.get("webqa_fluency_model", ""),
        )

    lb = rep.get("leaderboard_summary") or {}
    labels = {"All": "전체", "Unimodal": "단일모달", "Multimodal": "멀티모달"}
    for bucket in ("All", "Unimodal", "Multimodal"):
        b = lb.get(bucket) or {}
        if not b:
            continue
        print(
            f"  [{labels[bucket]}]  QA-FL={_f(b.get('QA-FL')):.4f}  "
            f"QA-Acc={_f(b.get('QA-Acc')):.4f}  QA={_f(b.get('QA')):.4f}"
        )

    scores = rep.get("scores") or {}
    all_s = scores.get("All") or {}
    if all_s:
        print(
            "원시 list F1 / list EM (전체):",
            f"{_f(all_s.get('f1')):.4f}",
            "/",
            f"{_f(all_s.get('em')):.4f}",
        )
        print(
            "키워드 list F1 / list EM (전체):",
            f"{_f(all_s.get('list_f1_keyword')):.4f}",
            "/",
            f"{_f(all_s.get('list_em_keyword')):.4f}",
        )
        if all_s.get("webqa_acc_approx") is not None:
            print(f"WebQA acc 근사 (전체): {_f(all_s.get('webqa_acc_approx')):.4f}")

    retr = rep.get("retrieval_summary") or {}
    if retr.get("status") == "missing_rankings_json":
        print("검색 지표: retrieval_rankings.json 없음 (해당 시 result/<RUN_ID>/에 배치)")
    elif retr and not retr.get("status"):
        k = int(retr.get("k_gen", 10))
        overall = retr.get("overall") or {}
        metrics_block = overall.get("metrics") or {}
        m = metrics_block.get(f"k={k}")
        if m:
            print(
                f"검색@{k} (전체)  hit@k={_f(m.get('hit@k')):.4f}  "
                f"recall@k={_f(m.get('recall@k')):.4f}  "
                f"MRR={_f(m.get('mrr@k')):.4f}  "
                f"retrieval F1={_f(m.get('retrieval_f1')):.4f}"
            )

    by_acc = scores.get("by_Qcate_webqa_acc_approx") or {}
    if by_acc:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_acc.items())]
        print("Qcate별 WebQA acc 근사:", " | ".join(parts))
    by_qa = scores.get("by_Qcate_webqa_qa") or {}
    if by_qa:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_qa.items())]
        print("Qcate별 QA(리더보드):", " | ".join(parts))

    print("상세 JSON:", report_path.resolve())
    print("=" * 62 + "\n")


if pred_json.is_file():
    report_json = phase5_dir / "evaluation_report.json"
    eval_env = {**common_env, "MMGRAPHRAG_RUN_ID": RUN_ID}
    _rc_p6 = run_step(
        "6단계 평가",
        [
            sys.executable,
            "eval/evaluate_webqa_qa.py",
            "--predictions",
            str(pred_json),
            "--gold_jsonl",
            str(q_file),
            "--report_json",
            str(report_json),
            "--split_label",
            "val",
        ],
        eval_env,
    )
    if _rc_p6 == 0:
        _print_webqa_report_summary_kr(report_json)
    else:
        print("평가 스크립트 비정상 종료 — 위쪽 영문 로그를 확인하세요. 종료 코드:", _rc_p6)
else:
    print("평가 건너뜀: predictions.json 없음")


## 부록: 터미널용 원시 셸 명령(선택)

노트북에서는 **단계별 실행(수동)** 셀이 권장입니다. 아래는 터미널만 쓸 때의 **bash** 예시입니다. `$RUN_ID`, 경로, 제한 값은 환경에 맞게 바꾸세요.

**0단계 — 슬라이스 내보내기** (`webqa_slice`를 이미 `result/$RUN_ID/`에 복사했다면 생략):
```bash
export MMGRAPHRAG_RUN_ID=my_run WEBQA_EXPORT_MAX=5 WEBQA_SLICE_DIR=result/my_run/webqa_slice
python export_webqa_slice.py
```

**2단계 — 패턴:**
```bash
export PATTERN_MAX_SAMPLES=5 PATTERN_CACHE_DIR=result/my_run/phase2_pattern_cache
python pattern.py
```

**3단계 — 추출:**
```bash
export EXTRACTION_QUESTION_FILE=result/my_run/webqa_slice/webqa_questions.jsonl ...
python extraction.py
```

환경 변수 배선은 `tests/test_pipeline.py`와 동일합니다 — 재현하려면 거기서 복사하면 됩니다.

## 데모 UI(선택)

`result/<RUN_ID>/`에 예측과 그래프가 있으면 `demo/be/config/paths.yaml`을 설정합니다(보통 `run_id: latest`). 자세한 내용은 `demo/README.md`.

**터미널 명령**이나 아래 **데모 BE+FE 시작** 셀로 노트북에서 서버를 띄울 수 있습니다(백그라운드 `subprocess`). 필요 조건:

- **BE:** 파이프라인과 같은 Python venv; 포트 **8000**(환경 변수 `DEMO_BE_PORT`로 변경).
- **FE:** **Node.js + npm**(예: nvm LTS); 포트 **5173**(`DEMO_FE_PORT`). Vite가 `/api`를 백엔드로 프록시합니다.

브라우저: **http://127.0.0.1:5173/** (FE) — API 상태: **http://127.0.0.1:8000/health**.

### 터미널에서 동일하게

```bash
cd demo/be && python server.py --host 0.0.0.0 --port 8000
cd demo/fe && npm install && npm run dev
```

### 노트북에서 데모 BE + FE 시작

서버를 **백그라운드**로 띄워 노트북을 계속 쓸 수 있습니다. 먼저 **저장소 루트와 경로** 셀을 실행해 `REPO`가 정의되게 하세요(없으면 현재 작업 디렉터리로 추정).

이 셀을 다시 실행하면 이 세션에서 이전에 띄운 서버(`DEMO_PROCESSES`)를 먼저 종료합니다.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "demo" / "be" / "server.py").is_file() else _here.parent

be_dir = _repo / "demo" / "be"
fe_dir = _repo / "demo" / "fe"
if not (be_dir / "server.py").is_file():
    raise FileNotFoundError(f"데모 백엔드를 찾을 수 없음: {be_dir}")

if "DEMO_PROCESSES" not in globals():
    DEMO_PROCESSES = []
else:
    for _p in list(DEMO_PROCESSES):
        if _p.poll() is None:
            _p.terminate()
    DEMO_PROCESSES.clear()

BE_PORT = int(os.environ.get("DEMO_BE_PORT", "8000"))
FE_PORT = int(os.environ.get("DEMO_FE_PORT", "5173"))

be_env = {**os.environ, "PYTHONPATH": str(be_dir)}
p_be = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", str(BE_PORT)],
    cwd=str(be_dir),
    env=be_env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
DEMO_PROCESSES.append(p_be)
print(f"데모 BE 시작 | pid={p_be.pid} | http://127.0.0.1:{BE_PORT}/health")
time.sleep(1.5)

npm = shutil.which("npm")
if npm is None:
    print("PATH에 npm 없음 — Node.js 설치 후: cd demo/fe && npm install && npm run dev")
else:
    if not (fe_dir / "node_modules").is_dir():
        print("demo/fe에서 npm install 실행 중(처음은 1분 정도 걸릴 수 있음)...")
        subprocess.run([npm, "install"], cwd=str(fe_dir), check=False)
    p_fe = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=str(fe_dir),
        env={**os.environ},
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    DEMO_PROCESSES.append(p_fe)
    print(f"데모 FE 시작 | pid={p_fe.pid} | http://127.0.0.1:{FE_PORT}/")

print("\n위 프론트 URL을 브라우저에서 엽니다. 종료는 다음 **데모 서버 중지** 셀을 실행하세요.")

### 데모 서버 중지

**데모 BE + FE 시작**에서 기록한 `DEMO_PROCESSES`를 종료합니다. 8000/5173이 여전히 열려 있으면 터미널에서 직접 중지하세요(`kill`, Windows는 작업 관리자).

In [ ]:
import time as _time

if "DEMO_PROCESSES" in globals() and DEMO_PROCESSES:
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.terminate()
    _time.sleep(0.5)
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.kill()
    DEMO_PROCESSES.clear()
    print("데모 BE/FE 프로세스를 종료했습니다.")
else:
    print("중지할 DEMO_PROCESSES 없음(먼저 데모 BE + FE 시작을 실행하세요).")

## 더 읽을 거리

- `README.md` — 개요, 데이터 구조, 모델 다운로드, 파이프라인 요약.
- `tests/test_pipeline.py` — 전체 오케스트레이션과 단계별 환경 변수.